In [ ]:
import os, sys, subprocess

# ── Clone repository ──
REPO_URL = "https://github.com/VvS-2403/SAiDL-Summer-Assignment-2026.git"
REPO_DIR = "/content/SAiDL-Summer-Assignment-2026"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}")

# ── Install dependencies ──
req_path = os.path.join(REPO_DIR, "requirements.txt")
if os.path.exists(req_path):
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_path], check=True)

# ── Make imports work ──
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")
print("Setup complete.")


In [ ]:
import os
try:
    import wandb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

api_key = os.environ.get("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key)
    print("WandB logged in via API key.")
else:
    wandb.login()
    print("WandB logged in interactively.")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilgpt2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    output_attentions=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(DEVICE)
model.eval()

sample_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "In the future, language models will help scientists discover new ideas.",
    "Causal interventions help identify which parts of a network matter.",
]
inputs = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
print(f"Model loaded on {DEVICE}. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


In [ ]:
def make_replacement_hook(replacement_tensor):
    def hook(module, input_, output):
        if isinstance(output, tuple):
            return (replacement_tensor,) + output[1:]
        return replacement_tensor
    return hook

def fake_quantize(tensor, bits=8):
    """Simulated uniform affine quantization."""
    q_min, q_max = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    t_min, t_max = tensor.min(), tensor.max()
    scale = (t_max - t_min).clamp(min=1e-8) / (q_max - q_min)
    zp = (q_min - (t_min / scale).round()).clamp(q_min, q_max)
    q = (tensor / scale + zp).round().clamp(q_min, q_max)
    return (q - zp) * scale

def patch_layer_and_run(model, tokenizer, text_batch, layer_index, quant_func):
    inp = tokenizer(text_batch, return_tensors="pt", truncation=True, padding=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        clean_out = model(**inp, output_hidden_states=True)
    clean_layer = clean_out.hidden_states[layer_index].detach()
    patched = quant_func(clean_layer)
    handle = model.transformer.h[layer_index - 1].register_forward_hook(
        make_replacement_hook(patched)
    )
    with torch.no_grad():
        out = model(**inp)
    handle.remove()
    return out.logits, clean_layer, patched

logits, clean_h, patched_h = patch_layer_and_run(
    model, tokenizer, sample_texts, layer_index=3,
    quant_func=lambda x: fake_quantize(x, bits=8),
)
print(f"Clean hidden shape: {clean_h.shape}")
print(f"Patched logits shape: {logits.shape}")
print(f"MSE from 8-bit patching: {((clean_h - patched_h)**2).mean().item():.6f}")


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)
    attentions = outputs.attentions

print(f"Extracted {len(attentions)} attention layers.")

layer_to_plot, head_to_plot = 0, 0
attn_weights = attentions[layer_to_plot][0, head_to_plot].float().cpu().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(attn_weights, cmap="viridis")
plt.title(f"Layer {layer_to_plot + 1}, Head {head_to_plot + 1} Attention")
plt.xlabel("Key Position")
plt.ylabel("Query Position")
plt.colorbar(label="Attention weight")
plt.tight_layout()
plt.show()


In [ ]:
hidden_states = outputs.hidden_states

for idx, hidden in enumerate(hidden_states):
    print(f"Layer {idx}: shape {hidden.shape}")

# Per-position activation norms at layer 3
layer_acts = hidden_states[3][0].detach().float().cpu().numpy()
mean_activation = np.linalg.norm(layer_acts, axis=-1)

plt.figure(figsize=(8, 3))
plt.plot(mean_activation, marker="o")
plt.title("Activation norm by token position \u2014 Layer 3")
plt.xlabel("Token position")
plt.ylabel("Activation norm")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
def ablate_neurons(layer_tensor, dims):
    patched = layer_tensor.clone()
    patched[..., dims] = 0.0
    return patched

logits_baseline, _, _ = patch_layer_and_run(model, tokenizer, sample_texts, layer_index=3, quant_func=lambda x: x)
logits_ablate, _, _ = patch_layer_and_run(model, tokenizer, sample_texts, layer_index=3, quant_func=lambda x: ablate_neurons(x, list(range(10))))

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else -100)
inputs_eval = tokenizer(sample_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
labels = inputs_eval.input_ids[:, 1:]

loss_base = loss_fn(logits_baseline[:, :-1, :].reshape(-1, logits_baseline.size(-1)), labels.reshape(-1)).item()
loss_ablate = loss_fn(logits_ablate[:, :-1, :].reshape(-1, logits_ablate.size(-1)), labels.reshape(-1)).item()

print(f"Baseline loss: {loss_base:.4f}")
print(f"Ablated loss (dims 0-9): {loss_ablate:.4f}")
print(f"Delta: {loss_ablate - loss_base:.4f}")


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

layer_norms = [hidden_states[i][0].detach().float().norm(dim=-1).mean().item() for i in range(len(hidden_states))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(layer_norms, marker="o")
ax1.set_title("Mean token representation norm across layers")
ax1.set_xlabel("Layer")
ax1.set_ylabel("Mean norm")
ax1.grid(True)

rep_matrix = np.stack([hidden_states[i][0].mean(dim=0).detach().float().cpu().numpy() for i in range(len(hidden_states))])
similarity = cosine_similarity(rep_matrix)

im = ax2.imshow(similarity, cmap="coolwarm", vmin=0, vmax=1)
ax2.set_title("Layer-wise cosine similarity")
ax2.set_xlabel("Layer")
ax2.set_ylabel("Layer")
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()


In [ ]:
import glob
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

SAVE_DIR = Path(REPO_DIR) / "mechanistic_interpretability" / "data" / "samples"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 128
MAX_TOKENS = 200_000  # configurable (200k debug -> 80M full)
FLUSH_EVERY = 500_000
TARGET_LAYER = 3

print(f"Saving activations to: {SAVE_DIR}")
print(f"Target tokens: {MAX_TOKENS:,}")

try:
    dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
except Exception as exc:
    raise RuntimeError("Failed to load openwebtext.") from exc

def flush_buffer(act_buf, tok_buf, idx):
    if not act_buf: return [], []
    stacked_acts = torch.cat(act_buf, dim=0)
    stacked_tokens = torch.cat(tok_buf, dim=0)
    path = SAVE_DIR / f"acts_{idx:04d}.pt"
    torch.save({"acts": stacked_acts.half(), "token_ids": stacked_tokens}, path)
    print(f"\nFlushed {stacked_acts.shape[0]:,} activations to {path}")
    return [], []

activation_buffer = []
token_id_buffer = []
token_buffer = []
total_extracted = 0
file_idx = 0

with torch.no_grad():
    for sample in tqdm(dataset, desc="Extracting"):
        text = sample.get("text", "")
        if not text:
            continue
        ids = tokenizer(text, add_special_tokens=False, truncation=True, max_length=model.config.n_ctx)["input_ids"]
        if not ids:
            continue
        token_buffer.extend(ids)

        while len(token_buffer) >= SEQ_LEN:
            chunk = token_buffer[:SEQ_LEN]
            token_buffer = token_buffer[SEQ_LEN:]
            x = torch.tensor([chunk], dtype=torch.long, device=DEVICE)

            outputs = model(x)
            h = outputs.hidden_states[TARGET_LAYER]
            h = h / (h.norm(dim=-1, keepdim=True) + 1e-8)

            activation_buffer.append(h.view(-1, 768).float().cpu())
            token_id_buffer.append(torch.tensor(chunk, dtype=torch.long))
            total_extracted += SEQ_LEN

            if len(activation_buffer) * SEQ_LEN >= FLUSH_EVERY:
                activation_buffer, token_id_buffer = flush_buffer(activation_buffer, token_id_buffer, file_idx)
                file_idx += 1

            if total_extracted >= MAX_TOKENS:
                break
        if total_extracted >= MAX_TOKENS:
            break

if activation_buffer:
    flush_buffer(activation_buffer, token_id_buffer, file_idx)

print(f"\nExtraction complete. Total tokens: {total_extracted:,}")


In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from mechanistic_interpretability.models.sae import TopKSparseAutoencoder

BOTTLENECK = 512
K = int(0.10 * BOTTLENECK)  # 51
LR = 1e-4
BATCH_SIZE = 4096
TARGET_STEPS = 100_000

sae = TopKSparseAutoencoder(d_model=768, d_sae=BOTTLENECK, k=K).to(DEVICE)
optimizer = optim.Adam(sae.parameters(), lr=LR)

act_files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))
assert act_files, "No activation files found. Run the extraction cell first."
print(f"Found {len(act_files)} activation files. Training SAE (m={BOTTLENECK}, k={K})...")

sae.train()
global_step = 0

for epoch in range(999):
    for path in act_files:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        activations = (payload["acts"] if isinstance(payload, dict) and "acts" in payload else payload).float()
        ds = TensorDataset(activations)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

        for (x,) in loader:
            x = x.to(DEVICE)
            x_recon, feats, l2_loss = sae(x)
            loss = l2_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
            optimizer.step()
            sae.set_decoder_norm_to_unit_norm()

            global_step += 1
            if global_step % 500 == 0:
                l0 = (feats > 0).float().sum(-1).mean().item()
                var_exp = 1.0 - (l2_loss.item() / x.var().item())
                print(f"step {global_step:>6d} | loss={loss.item():.4f} | l0={l0:.1f} | var_exp={var_exp:.3f}")
            if global_step >= TARGET_STEPS:
                break
        if global_step >= TARGET_STEPS:
            break
    if global_step >= TARGET_STEPS:
        break

SAE_PATH = Path(REPO_DIR) / "mechanistic_interpretability" / "outputs" / f"sae_m{BOTTLENECK}_k{K}.pt"
SAE_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(sae.state_dict(), SAE_PATH)
print(f"\nSaved SAE to {SAE_PATH} after {global_step} steps.")


In [ ]:
import math
import json
import umap
from mechanistic_interpretability.utils.metrics import compute_sds, linear_cka

def quantise(h, bits, mode="per_tensor"):
    q_min, q_max = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    if mode == "per_tensor":
        t_min, t_max = h.min(), h.max()
    else:
        t_min = h.reshape(-1, h.shape[-1]).min(0).values
        t_max = h.reshape(-1, h.shape[-1]).max(0).values
        t_min = t_min.view(*([1] * (h.dim() - 1)), -1)
        t_max = t_max.view(*([1] * (h.dim() - 1)), -1)
    scale = (t_max - t_min).clamp(min=1e-8) / (q_max - q_min)
    zp = (q_min - (t_min / scale).round()).clamp(q_min, q_max)
    q = (h / scale + zp).round().clamp(q_min, q_max)
    return (q - zp) * scale

def compute_ppl_with_quantised_layer(model, tokenizer, text_batch, quant_fn, device, target_layer=3):
    import torch.nn.functional as F
    inp = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    input_ids = inp["input_ids"]
    with torch.no_grad():
        out_clean = model(**inp, output_hidden_states=True)
    h_clean = out_clean.hidden_states[target_layer].clone()
    h_quant = quant_fn(h_clean)
    handle = model.transformer.h[target_layer - 1].register_forward_hook(make_replacement_hook(h_quant))
    with torch.no_grad():
        out_quant = model(**inp)
    handle.remove()
    logits = out_quant.logits
    shift_logits = logits[:, :-1, :].reshape(-1, logits.size(-1))
    shift_labels = input_ids[:, 1:].reshape(-1)
    loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=tokenizer.pad_token_id or -100)
    return loss.item(), h_clean, h_quant

print("Collecting 10k activations for quantisation analysis...")
H_clean_list = []
with torch.no_grad():
    eval_dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True)
    for sample in tqdm(eval_dataset, desc="Collecting", total=300):
        ids = tokenizer(sample["text"], add_special_tokens=False)["input_ids"]
        if len(ids) < 128:
            continue
        chunk = ids[:128]
        x = torch.tensor([chunk], dtype=torch.long, device=DEVICE)
        out = model(x)
        h = out.hidden_states[3]
        h = h / (h.norm(dim=-1, keepdim=True) + 1e-8)
        H_clean_list.append(h.view(-1, 768).float().cpu())
        if len(H_clean_list) * 128 >= 10000:
            break

H_clean_10k = torch.cat(H_clean_list, dim=0)[:10000]
print(f"Collected: {H_clean_10k.shape}")

BITS_LIST = [8, 4, 2]
QUANT_MODES = ["per_tensor", "per_feature"]
results = {}

for bits in BITS_LIST:
    for mode in QUANT_MODES:
        key = f"{bits}bit_{mode}"
        print(f"\n=== {key} ===")

        H_quant_10k = quantise(H_clean_10k, bits, mode)
        mse = ((H_clean_10k - H_quant_10k) ** 2).mean().item()
        sds = compute_sds(H_clean_10k.numpy(), H_quant_10k.numpy(), k=32)
        cka = linear_cka(H_clean_10k[:1000].numpy(), H_quant_10k[:1000].numpy())

        # PPL via replacement hook (use validation split)
        eval_loss, eval_count = 0.0, 0
        eval_ds = load_dataset("openwebtext", split="test", streaming=True, trust_remote_code=True)
        # Using a dummy split or taking next samples from train if test is not easily streamable
        # openwebtext typically doesn't have 'test' split. We'll use 'train' and skip first 1000 samples for pseudo-validation.
        val_dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True).skip(1000)
        
        for sample in tqdm(val_dataset, desc=f"PPL {key}", total=20):
            loss, _, _ = compute_ppl_with_quantised_layer(
                model, tokenizer, [sample["text"]], lambda h: quantise(h, bits, mode), DEVICE
            )
            eval_loss += loss
            eval_count += 1
            if eval_count >= 20:
                break
        ppl = math.exp(eval_loss / max(eval_count, 1))

        results[key] = {"bits": bits, "mode": mode, "mse": mse, "sds": sds, "cka": cka, "ppl": ppl}
        print(f"{key}: MSE={mse:.6f}, SDS={sds:.4f}, CKA={cka:.4f}, PPL={ppl:.2f}")

        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
        emb_clean = reducer.fit_transform(H_clean_10k.numpy())
        emb_quant = reducer.transform(H_quant_10k.numpy())

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        ax1.scatter(emb_clean[:, 0], emb_clean[:, 1], s=1, alpha=0.4, c="steelblue")
        ax1.set_title("FP32 Baseline")
        ax1.axis("off")
        ax2.scatter(emb_quant[:, 0], emb_quant[:, 1], s=1, alpha=0.4, c="crimson")
        ax2.set_title(f"{bits}-bit {mode}")
        ax2.axis("off")
        plt.suptitle(f"UMAP: {key}", fontsize=14)
        plt.tight_layout()
        plt.show()

out_path = Path(REPO_DIR) / "mechanistic_interpretability" / "outputs" / "quant_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {out_path}")


In [ ]:
import math

TARGET_LAYER = 3

def rank_neurons_by_damage(H_clean, H_quant):
    l2_per_dim = ((H_clean - H_quant) ** 2).mean(0)
    mu_c, sig_c = H_clean.mean(0), H_clean.std(0).clamp(min=1e-8)
    mu_q, sig_q = H_quant.mean(0), H_quant.std(0).clamp(min=1e-8)
    kl_per_dim = (
        (sig_q / sig_c).log()
        + (sig_c ** 2 + (mu_c - mu_q) ** 2) / (2 * sig_q ** 2)
        - 0.5
    )
    return {
        "l2_per_dim": l2_per_dim,
        "kl_per_dim": kl_per_dim,
        "l2_rank": torch.argsort(l2_per_dim, descending=True),
        "kl_rank": torch.argsort(kl_per_dim, descending=True),
    }

def spectral_analysis(H_clean, H_quant, k=64, label="4bit"):
    H_c = (H_clean - H_clean.mean(0)).float()
    H_q = (H_quant - H_quant.mean(0)).float()
    _, S_c, Vc = torch.linalg.svd(H_c, full_matrices=False)
    _, S_q, Vq = torch.linalg.svd(H_q, full_matrices=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(S_c[:200].numpy(), label="FP32", lw=2)
    ax1.plot(S_q[:200].numpy(), label=label, lw=2, ls="--")
    ax1.legend()
    ax1.set_title(f"Singular Value Spectrum: FP32 vs {label}")
    ax1.set_xlabel("Index")
    ax1.set_ylabel("Singular value")

    M = Vc[:k] @ Vq[:k].T
    sv = torch.linalg.svdvals(M).clamp(-1, 1)
    angles_deg = torch.acos(sv) * 180 / torch.pi

    ax2.plot(angles_deg.numpy())
    ax2.set_title(f"Principal Angles: FP32 vs {label}")
    ax2.set_xlabel("Component index")
    ax2.set_ylabel("Angle (degrees)")
    plt.tight_layout()
    plt.show()

    sds = compute_sds(H_clean.numpy(), H_quant.numpy(), k=k)
    print(f"[{label}] SDS={sds:.4f}, mean_angle={angles_deg.mean().item():.2f} deg")
    return angles_deg

def top_activating_tokens_for_dims(dims, n_top=10, max_files=3):
    records = {dim: [] for dim in dims}
    files = sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))[:max_files]
    for path in files:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        acts = (payload["acts"] if isinstance(payload, dict) and "acts" in payload else payload).float()
        token_ids = payload["token_ids"]
        for dim in dims:
            values = acts[:, dim]
            topk = values.topk(min(n_top, len(values)))
            for score, idx in zip(topk.values.tolist(), topk.indices.tolist()):
                tok_id = int(token_ids[idx].item())
                records[dim].append((score, tokenizer.convert_ids_to_tokens([tok_id])[0]))
    for dim in records:
        records[dim] = sorted(records[dim], key=lambda x: x[0], reverse=True)[:n_top]
    return records

def ablation_ppl(model, tokenizer, text_batch, dims_to_zero, device, target_layer=3):
    def zero_hook(module, input_, output):
        h = output[0].clone() if isinstance(output, tuple) else output.clone()
        if dims_to_zero:
            h[:, :, dims_to_zero] = 0.0
        return (h,) + output[1:] if isinstance(output, tuple) else h
    handle = model.transformer.h[target_layer - 1].register_forward_hook(zero_hook)
    inp = tokenizer(text_batch, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    with torch.no_grad():
        out = model(**inp)
    handle.remove()
    ids = inp["input_ids"]
    logits = out.logits
    loss = torch.nn.functional.cross_entropy(
        logits[:, :-1, :].reshape(-1, logits.size(-1)),
        ids[:, 1:].reshape(-1),
        ignore_index=tokenizer.pad_token_id or -100,
    )
    return math.exp(loss.item())

H_q4 = quantise(H_clean_10k, 4, "per_tensor")
damage = rank_neurons_by_damage(H_clean_10k, H_q4)
top20_l2 = damage["l2_rank"][:20].tolist()
top20_kl = damage["kl_rank"][:20].tolist()
print("Top-20 L2-damaged dims:", top20_l2)
print("Top-20 KL-damaged dims:", top20_kl)

angles = spectral_analysis(H_clean_10k, H_q4, k=64, label="4bit_per_tensor")

records = top_activating_tokens_for_dims(top20_l2[:5], n_top=5)
print("\nTop activating tokens for top 5 L2-damaged dims:")
for dim, tokens in records.items():
    print(f"  dim {dim}: {tokens}")

eval_texts = [s["text"] for _, s in zip(range(10), load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True))]
TOP20 = top20_l2[:20]
RAND20 = torch.randperm(768)[:20].tolist()

ppl_baseline = ablation_ppl(model, tokenizer, eval_texts[:5], [], DEVICE)
ppl_top20 = ablation_ppl(model, tokenizer, eval_texts[:5], TOP20, DEVICE)
ppl_rand20 = ablation_ppl(model, tokenizer, eval_texts[:5], RAND20, DEVICE)

print(f"\nPPL baseline: {ppl_baseline:.2f}")
print(f"PPL top-20 L2-damaged ablated: {ppl_top20:.2f}  delta={ppl_top20 - ppl_baseline:.2f}")
print(f"PPL random-20 ablated: {ppl_rand20:.2f}  delta={ppl_rand20 - ppl_baseline:.2f}")

def per_dim_ablation_ppl_delta(model, tokenizer, eval_texts, n_dims=50, device=DEVICE):
    baseline = ablation_ppl(model, tokenizer, eval_texts, [], device)
    deltas = {}
    for dim in tqdm(range(n_dims), desc="Per-dim ablation"):
        deltas[dim] = ablation_ppl(model, tokenizer, eval_texts, [dim], device) - baseline
    return deltas

ppl_deltas = per_dim_ablation_ppl_delta(model, tokenizer, eval_texts[:5], n_dims=50)
ppl_rank = sorted(ppl_deltas, key=ppl_deltas.get, reverse=True)
overlap = len(set(TOP20) & set(ppl_rank[:20]))
print(f"\nOverlap: top-20 L2-damaged vs top-20 PPL-critical: {overlap}/20")


In [ ]:
import torch.nn.functional as F

# Prepare sample loader for Jacobian/Fisher analysis
sample_payload = torch.load(sorted(glob.glob(str(SAVE_DIR / "acts_*.pt")))[0], map_location="cpu", weights_only=False)
sample_acts = (sample_payload["acts"] if isinstance(sample_payload, dict) and "acts" in sample_payload else sample_payload).float()
sample_ds = TensorDataset(sample_acts[:2048])
sample_loader = DataLoader(sample_ds, batch_size=256, shuffle=False)


In [ ]:
print("Computing Jacobian norms for SAE encoder sensitivity...")
sae.eval()

N_UNITS_TO_PROBE = min(sae.d_sae, 200)
jacobian_norms = torch.zeros(N_UNITS_TO_PROBE)
n_batches = 0

for (x,) in sample_loader:
    x = x.to(DEVICE).requires_grad_(True)
    z = F.relu(sae.encoder(x - sae.b_dec))
    
    for unit_idx in range(N_UNITS_TO_PROBE):
        grad = torch.autograd.grad(z[:, unit_idx].sum(), x, retain_graph=True)[0]
        jacobian_norms[unit_idx] += grad.norm(dim=-1).mean().item()
        
    n_batches += 1

jacobian_norms /= max(n_batches, 1)
print("Jacobian norms for top 200 units:", jacobian_norms[:10])


In [ ]:
print("Computing Fisher importance scores for SAE bottleneck units...")
sae.eval()
fisher_scores = torch.zeros(sae.d_sae)
n_batches = 0

for (x,) in sample_loader:
    x = x.to(DEVICE)
    x.requires_grad_(False)
    z = F.relu(sae.encoder(x - sae.b_dec))
    z.requires_grad_(True)
    topk_vals, topk_idx = z.topk(sae.k, dim=-1)
    sparse_z = torch.zeros_like(z).scatter_(-1, topk_idx, topk_vals)
    recon = sae.decoder(sparse_z) + sae.b_dec
    loss = F.mse_loss(recon, x)
    
    grad_z = torch.autograd.grad(loss, z, create_graph=False)[0]
    fisher_scores += (grad_z ** 2).mean(0).cpu()
    n_batches += 1

fisher_scores /= max(n_batches, 1)
print("Fisher scores (first 10 units):", fisher_scores[:10])


In [ ]:
print("Testing for low-variance collapse...")
plt.figure(figsize=(10, 6))

for bits in [8, 4, 2]:
    H_q = quantise(H_clean_10k, bits, "per_tensor")
    range_fp = (H_clean_10k.max(0).values - H_clean_10k.min(0).values)
    range_q  = (H_q.max(0).values - H_q.min(0).values)
    ratio = range_q / (range_fp + 1e-8)
    collapsed = (ratio < 0.5).sum().item()
    print(f"[{bits}-bit] Collapsed dims (ratio < 0.5): {collapsed} / {H_clean_10k.shape[-1]}")
    
    plt.hist(ratio.numpy(), bins=50, alpha=0.5, label=f"{bits}-bit")

plt.legend()
plt.title("Effective Range Ratio (Quantised / FP32)")
plt.xlabel("Ratio")
plt.ylabel("Count")
plt.show()


In [ ]:
print("Measuring sparsity fragility...")
def get_topk_indices(sae_model, h_batch):
    z = F.relu(sae_model.encoder(h_batch.to(DEVICE) - sae_model.b_dec))
    return z.topk(sae_model.k, dim=-1).indices

def compute_jaccard(idx_clean, idx_quant):
    clean_sets = [set(row.tolist()) for row in idx_clean]
    quant_sets = [set(row.tolist()) for row in idx_quant]
    jaccards = []
    for s_c, s_q in zip(clean_sets, quant_sets):
        inter = len(s_c & s_q)
        union = len(s_c | s_q)
        jaccards.append(inter / union if union > 0 else 0.0)
    return torch.tensor(jaccards).mean().item()

H_clean_batch = H_clean_10k[:2048]
z_clean = get_topk_indices(sae, H_clean_batch)
z_quant = get_topk_indices(sae, quantise(H_clean_batch, 4, "per_tensor").to(DEVICE))
jaccard = compute_jaccard(z_clean, z_quant)
print(f"Jaccard similarity of active sets (FP32 vs 4-bit): {jaccard:.4f}")


In [ ]:
print("Testing subspace rotation...")
def sae_encode(h_batch):
    return F.relu(sae.encoder(h_batch.to(DEVICE) - sae.b_dec)).cpu()

z_clean_acts = sae_encode(H_clean_10k)
z_quant_acts = sae_encode(quantise(H_clean_10k, 4, "per_tensor"))

_, _, Vc = torch.linalg.svd(z_clean_acts - z_clean_acts.mean(0), full_matrices=False)
_, _, Vq = torch.linalg.svd(z_quant_acts - z_quant_acts.mean(0), full_matrices=False)

k_sub = min(64, sae.d_sae)
M = Vc[:k_sub] @ Vq[:k_sub].T
angles = torch.acos(torch.linalg.svdvals(M).clamp(-1, 1)) * 180 / torch.pi
print(f"Subspace Rotation Principal Angles (mean top-{k_sub}): {angles.mean().item():.2f} deg")


In [ ]:
print("Evaluating Subspace-Preserving Quantisation (SPQ)...")
class SubspacePreservingQuantiser:
    def __init__(self, H_fp32, k=64, bits_important=8, bits_residual=2):
        H_c = (H_fp32 - H_fp32.mean(0)).float()
        _, _, V = torch.linalg.svd(H_c, full_matrices=False)
        self.V_k = V[:k].T.to(DEVICE)
        self.bits_important = bits_important
        self.bits_residual = bits_residual
        self.mean = H_fp32.mean(0).to(DEVICE)
    
    def __call__(self, h):
        h = h.to(DEVICE)
        h_centered = h - self.mean
        coeffs = h_centered @ self.V_k
        h_imp = coeffs @ self.V_k.T
        h_res = h_centered - h_imp
        
        q_imp = quantise(h_imp.cpu(), self.bits_important, "per_tensor").to(DEVICE)
        q_res = quantise(h_res.cpu(), self.bits_residual, "per_tensor").to(DEVICE)
        
        return (q_imp + q_res + self.mean).cpu()

spq = SubspacePreservingQuantiser(H_clean_10k)
H_spq = spq(H_clean_10k)

sds_spq = compute_sds(H_clean_10k.numpy(), H_spq.numpy(), k=32)
sds_std4 = compute_sds(H_clean_10k.numpy(), quantise(H_clean_10k, 4, "per_tensor").numpy(), k=32)

cka_spq = linear_cka(H_clean_10k[:1000].numpy(), H_spq[:1000].numpy())
cka_std4 = linear_cka(H_clean_10k[:1000].numpy(), quantise(H_clean_10k, 4, "per_tensor")[:1000].numpy())

print(f"SPQ SDS: {sds_spq:.4f} | Std 4-bit SDS: {sds_std4:.4f}")
print(f"SPQ CKA: {cka_spq:.4f} | Std 4-bit CKA: {cka_std4:.4f}")

eval_loss_spq, eval_count = 0.0, 0
eval_loss_std4 = 0.0
val_dataset = load_dataset("openwebtext", split="train", streaming=True, trust_remote_code=True).skip(1000)

for sample in tqdm(val_dataset, desc="PPL SPQ vs 4-bit", total=20):
    loss_spq, _, _ = compute_ppl_with_quantised_layer(
        model, tokenizer, [sample["text"]], spq, DEVICE
    )
    loss_std4, _, _ = compute_ppl_with_quantised_layer(
        model, tokenizer, [sample["text"]], lambda h: quantise(h, 4, "per_tensor"), DEVICE
    )
    eval_loss_spq += loss_spq
    eval_loss_std4 += loss_std4
    eval_count += 1
    if eval_count >= 20:
        break

ppl_spq = math.exp(eval_loss_spq / max(eval_count, 1))
ppl_std4 = math.exp(eval_loss_std4 / max(eval_count, 1))
print(f"SPQ PPL: {ppl_spq:.2f} | Std 4-bit PPL: {ppl_std4:.2f}")


In [ ]:
print("Training SAE (m=1024)...")
BOTTLENECK_1024 = 1024
K_1024 = int(0.10 * BOTTLENECK_1024)  # 102

sae_1024 = TopKSparseAutoencoder(d_model=768, d_sae=BOTTLENECK_1024, k=K_1024).to(DEVICE)
optimizer_1024 = optim.Adam(sae_1024.parameters(), lr=LR)

sae_1024.train()
global_step_1024 = 0

for epoch in range(999):
    for path in act_files:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        activations = (payload["acts"] if isinstance(payload, dict) and "acts" in payload else payload).float()
        ds = TensorDataset(activations)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

        for (x,) in loader:
            x = x.to(DEVICE)
            x_recon, feats, l2_loss = sae_1024(x)
            loss = l2_loss

            optimizer_1024.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(sae_1024.parameters(), 1.0)
            optimizer_1024.step()
            sae_1024.set_decoder_norm_to_unit_norm()

            global_step_1024 += 1
            if global_step_1024 % 500 == 0:
                l0 = (feats > 0).float().sum(-1).mean().item()
                var_exp = 1.0 - (l2_loss.item() / x.var().item())
                print(f"step {global_step_1024:>6d} | loss={loss.item():.4f} | l0={l0:.1f} | var_exp={var_exp:.3f}")
            if global_step_1024 >= TARGET_STEPS:
                break
        if global_step_1024 >= TARGET_STEPS:
            break
    if global_step_1024 >= TARGET_STEPS:
        break

SAE_PATH_1024 = Path(REPO_DIR) / "mechanistic_interpretability" / "outputs" / f"sae_m{BOTTLENECK_1024}_k{K_1024}.pt"
SAE_PATH_1024.parent.mkdir(parents=True, exist_ok=True)
torch.save(sae_1024.state_dict(), SAE_PATH_1024)
print(f"\nSaved SAE to {SAE_PATH_1024} after {global_step_1024} steps.")
